In [2]:
import sys
print(sys.executable)

c:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\python.exe


# Importacion de librerías, definición de raíz del proyecto y descarga de datos (setup)

In [3]:
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import requests
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegressionCV

Vamos a definir la raíz del proyecto y evitar rutas absolutas.

In [4]:
load_dotenv(find_dotenv())

PROJECT_ROOT = Path(find_dotenv()).parent.resolve()
print(f"PROJECT_ROOT set to: {PROJECT_ROOT}")

DATA_DIR = PROJECT_ROOT / 'data'

PROJECT_ROOT set to: C:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025


Comprobamos que el archivo existe y si no es así lo descargamos automáticamente.

In [5]:
DATA_URL = "https://data.mendeley.com/public-files/datasets/b24cb4nn8h/files/94aa2cf6-b4c1-4140-bfcc-d21040a14735/file_downloaded"
FILE_PATH = DATA_DIR / "synthea_30k.csv"

DATA_DIR.mkdir(parents = True ,exist_ok = True )

if not FILE_PATH.exists():
    print("Descargando dataset...")
    response = requests.get(DATA_URL)
    with open(FILE_PATH, 'wb') as f:
        f.write(response.content)
    print("Descarga completada.")
else:
    print("Dataset ya disponible localmente.")

Dataset ya disponible localmente.


# Creación de DataFrame y procesamiento de los datos

In [6]:
datos = pd.read_csv(FILE_PATH, low_memory=False)

In [7]:
datos.shape

(4221, 765)

In [8]:
datos.head()

,ptnum,label,scc,C-44465007,C-392091000,C-248595008,C-6246-3,C-197927001,C-234262008,C-389221,...,C-274804006,C-38822007,C-704-7,C-72514-3,C-6844-5,C-268556000,C-979492,C-122548005,C-141918,C-1359133
0,p15865,1,150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
1,p27452,1,145,True,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,NaN,NaN
2,p12382,1,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
3,p21814,1,132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
4,p2902,1,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,NaN,NaN


Lo primero que nos llama la atención es la cantidad de datos faltantes y la enorme cantidad de variables. Para entender mejor cuales son estas variables recurrimos al repositorio GitHub del autor (https://github.com/lhs-open/synthetic-data/blob/main/lung-cancer-risk/synthea-lc-dataset-codes.csv) Donde en el archivo csv, da nombre a cada una de las variables. La codificación de estos nombres es la habitual en proyectos médicos y es el original standard code (SNOMED-CT, LOINC, etc.) precedido por 'C-' (Información extraida del readme del repositorio).

Gracias a esta información nos damos cuenta que es normal que falten tantos datos y es que se recoge información de enfermedades tan espécificas como un esguice de tobillo, la cual la mayoría de paciente no ha sufrido o puede que la haya sufrido pero no este recogido en los datos.

Sea como sea, es inviable trabajar con tantas variables con datos faltantes, haremos primero la selección de las 50 mejores variables para trabajar de una manera más eficiente. Pero antes, debemos de identificar la variable objetivo.

In [9]:
columnas_C = datos.filter(regex='^C-')
datos.drop(columnas_C,axis=1)

,ptnum,label,scc
0,p15865,1,150
1,p27452,1,145
2,p12382,1,133
3,p21814,1,132
4,p2902,1,127
...,...,...,...
4216,p9457,0,100
4217,p967,0,100
4218,p9736,0,100
4219,p9788,0,100


Extrayendo todas las columnas que empiezan por "C-". Nos quedamos únicamente con tres atributos, el identificador del paciente, una variable binaria y lo que parece ser otra de las variables explicativas `scc`. En este punto es fácil detectar que la variable que mide la presencia o no de cancér de pulmón es `label`, es decir, nuestra variable objetivo (target). 

In [10]:
datos=datos.rename(columns={'label': 'target'})
datos=datos.set_index("ptnum")

In [11]:
X= datos.drop("target",axis=1)
y=datos["target"]
X.head()

,scc,C-44465007,C-392091000,C-248595008,C-6246-3,C-197927001,C-234262008,C-389221,C-92691004,C-392021009,...,C-274804006,C-38822007,C-704-7,C-72514-3,C-6844-5,C-268556000,C-979492,C-122548005,C-141918,C-1359133
ptnum,,,,,,,,,,,,,,,,,,,,,
p15865,150,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
p27452,145,True,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,NaN,NaN
p12382,133,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
p21814,132,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,abnormal,NaN,NaN,NaN,NaN,NaN,NaN
p2902,127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True,NaN,...,NaN,NaN,NaN,normal,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
from sklearn.model_selection import train_test_split
datos_train,datos_test= train_test_split(datos.drop("scc",axis=1), test_size=0.2, random_state=625,stratify=y)

In [13]:
X_train= datos_train.drop("target",axis=1)
y_train=datos_train["target"]

In [ ]:
datos_train.select_dtypes(exclude=['object','float'])

,target
ptnum,
p24407,0
p3961,0
p21560,1
p28259,0
p19603,0
...,...
p12637,0
p6616,0
p6702,0


: 

In [64]:
from sklearn.compose import make_column_selector, ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.feature_selection import SelectKBest
from sklearn.impute import SimpleImputer


cat_pipeline = Pipeline([
    ("codif", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ("imputer",SimpleImputer(strategy="constant", fill_value=0.0,keep_empty_features=True)),
])

preprocesamiento = ColumnTransformer([
    ("cat", cat_pipeline, make_column_selector(dtype_include="object")),
], remainder=(SimpleImputer(strategy="constant", fill_value=0.0)), verbose_feature_names_out=False)


In [65]:
X_trans=pd.DataFrame(preprocesamiento.fit_transform(X_train),columns=preprocesamiento.get_feature_names_out(),index=X_train.index)
X_trans.head()

c:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['C-26881-3' 'C-79733001' 'C-789980' 'C-309845' 'C-448417001'
 'C-171231001' 'C-48387007' 'C-979092' 'C-82078001' 'C-60951000119105'
 'C-200936003' 'C-104173009' 'C-698423002' 'C-448813005' 'C-14152002'
 'C-256277009' 'C-288328004' 'C-200243' 'C-1940648' 'C-408512008'
 'C-48333001' 'C-727711' 'C-281790008' 'C-113076002' 'C-782576004'
 'C-999998' 'C-2284960' 'C-1483744' 'C-864706' 'C-1373463']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


,C-44465007,C-392091000,C-248595008,C-6246-3,C-197927001,C-234262008,C-389221,C-92691004,C-392021009,C-308182,...,C-30385-9,C-33037-3,C-26453-1,C-1960-4,C-751-8,C-731-0,C-6301-6,C-75443-2,C-26464-8,C-704-7
ptnum,,,,,,,,,,,,,,,,,,,,,
p24407,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p3961,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p21560,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p28259,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p19603,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
X_trans.describe()

,C-44465007,C-392091000,C-248595008,C-6246-3,C-197927001,C-234262008,C-389221,C-92691004,C-392021009,C-308182,...,C-30385-9,C-33037-3,C-26453-1,C-1960-4,C-751-8,C-731-0,C-6301-6,C-75443-2,C-26464-8,C-704-7
count,3376.0,3376.0,3376.0,3376.0,3376.0,3376.0,3376.0,3376.0,3376.0,3376.0,...,3376.000000,3376.000000,3376.000000,3376.000000,3376.000000,3376.000000,3376.000000,3376.000000,3376.000000,3376.000000
mean,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.062500,0.044017,0.024615,0.232044,0.055181,0.019624,0.047497,0.000889,0.061345,0.006117
std,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.908699,0.674399,0.357930,2.372796,0.380782,0.136595,0.341526,0.029801,0.893367,0.042145
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,14.500000,14.900000,5.900000,25.570000,3.260000,1.090000,4.160000,1.000000,14.800000,0.340000


In [67]:
seleccion = SelectKBest(k=50)
seleccion.set_output(transform="pandas")
X_seleccion = seleccion.fit_transform(X_trans,y_train)

c:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [  0   1   2   3   4   5   6   7   8   9  10  11  12  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  34  35  37  38
  39  42  43  44  45  46  47  48  49  51  52  53  54  57  58  59  61  62
  63  64  65  66  67  68  69  70  71  72  73  74  75  76  77  80  81  82
  83  84  85  86  87  88  90  91  92  93  94  95  96  97  98  99 100 101
 102 103 104 105 106 107 108 110 111 112 113 114 115 116 117 118 119 120
 121 122 123 124 125 126 127 128 129 130 132 133 134 135 136 137 138 139
 140 141 142 143 144 145 146 147 148 149 150 151 152 153 154 155 156 157
 158 159 161 162 163 164 165 166 167 168 169 172 173 174 175 176 177 178
 179 180 181 182 183 184 185 188 189 190 191 192 193 194 195 196 197 198
 199 200 201 202 203 204 205 206 207 208 209 210 211 212 214 215 21

In [68]:
X_seleccion.shape

(3376, 50)

In [69]:
X_seleccion.head()

,C-2500-7,C-6768-6,C-424144002,C-6690-2,C-1742-6,C-1920-8,C-4548-4,C-2069-3,C-88020-3,C-85337-4,...,C-770-8,C-742-7,C-5902-2,C-706-2,C-1988-5,C-736-9,C-713-8,C-5905-5,C-751-8,C-704-7
ptnum,,,,,,,,,,,,,,,,,,,,,
p24407,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p3961,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p21560,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p28259,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
p19603,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [40]:
X_seleccion.nunique()

C-2500-7        2
C-6768-6        2
C-424144002     3
C-6690-2        2
C-1742-6        2
C-1920-8        2
C-4548-4        2
C-2069-3        2
C-88020-3       4
C-85337-4       2
C-788-0         2
C-789-8         2
C-18262-6       2
C-718-7         2
C-2345-7        2
C-33914-3       2
C-777-3         2
C-94531-1       2
C-89579-7       2
C-46288-7       2
C-19123-9       2
C-10230-1       2
C-21000-5       2
C-4544-3        2
C-2276-4        2
C-2502-3        2
C-2339-0        2
C-1751-7        2
C-2028-9        2
C-72166-2       2
C-2160-0        2
C-20565-8       2
C-1975-2        2
C-2857-1        2
C-2498-4        2
C-2075-0        2
C-88021-1       3
C-785-6         2
C-263495000     2
C-711-2        18
C-770-8        69
C-742-7        28
C-5902-2       58
C-706-2        44
C-1988-5       58
C-736-9        67
C-713-8        45
C-5905-5       64
C-751-8        49
C-704-7        12
dtype: int64

In [70]:
modelo= LogisticRegressionCV(max_iter=1000, n_jobs=-1, scoring="recall", cv=5)
modelo.fit(X_seleccion,y_train)

c:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\Lib\site-packages\sklearn\linear_model\_logistic.py:1780: FutureWarning: The default value for l1_ratios will change from None to (0.0,) in version 1.10. From version 1.10 onwards, only array-like with values in [0, 1] will be allowed, None will be forbidden. To avoid this warning, explicitly set a value, e.g. l1_ratios=(0,).
  warnings.warn(
c:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True

,"Cs Cs: int or list of floats, default=10Each of the values in Cs describes the inverse of regularizationstrength. If Cs is as an int, then a grid of Cs values are chosenin a logarithmic scale between 1e-4 and 1e4.Like in support vector machines, smaller values specify strongerregularization.",10
,"l1_ratios l1_ratios: array-like of shape (n_l1_ratios), default=NoneFloats between 0 and 1 passed as Elastic-Net mixing parameter (scaling betweenL1 and L2 penalties). For `l1_ratio = 0` the penalty is an L2 penalty. For`l1_ratio = 1` it is an L1 penalty. For `0 < l1_ratio < 1`, the penalty is acombination of L1 and L2.All the values of the given array-like are tested by cross-validation and theone giving the best prediction score is used... warning:: Certain values of `l1_ratios`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `l1_ratios=None` is deprecated in 1.8 and will raise an error in version 1.10. Default value will change from `None` to `(0.0,)` in version 1.10.",'warn'
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"cv cv: int or cross-validation generator, default=NoneThe default cross-validation generator used is Stratified K-Folds.If an integer is provided, it specifies the number of folds, `n_folds`, used.See the module :mod:`sklearn.model_selection` module for thelist of possible cross-validation objects... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer dual=False whenn_samples > n_features.",False
,"penalty penalty: {'l1', 'l2', 'elasticnet'}, default='l2'Specify the norm of the penalty:- `'l2'`: add a L2 penalty term (used by default);- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"scoring scoring: str or callable, default=NoneThe scoring method to use for cross-validation. Options:- str: see :ref:`scoring_string_names` for options.- callable: a scorer callable object (e.g., function) with signature ``scorer(estimator, X, y)``. See :ref:`scoring_callable` for details.- `None`: :ref:`accuracy ` is used.",'recall'
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' might be slower in :class:`LogisticRegressionCV` because it does not handle warm-starting.- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass

In [71]:
X_test= datos_test.drop("target",axis=1)
y_test=datos_test["target"]
X_test_trans= pd.DataFrame(preprocesamiento.transform(X_test),columns=preprocesamiento.get_feature_names_out(),index=X_test.index)
X_test_seleccion= seleccion.transform(X_test_trans)

c:\Users\rtrin\OneDrive\Escritorio\FORMACIÓN\MÁSTER\TFM\tfm-mulcia-2025\tfm-mulcia-2025\.pixi\envs\default\Lib\site-packages\sklearn\impute\_base.py:641: UserWarning: Skipping features without any observed values: ['C-26881-3' 'C-79733001' 'C-789980' 'C-309845' 'C-448417001'
 'C-171231001' 'C-48387007' 'C-979092' 'C-82078001' 'C-60951000119105'
 'C-200936003' 'C-104173009' 'C-698423002' 'C-448813005' 'C-14152002'
 'C-256277009' 'C-288328004' 'C-200243' 'C-1940648' 'C-408512008'
 'C-48333001' 'C-727711' 'C-281790008' 'C-113076002' 'C-782576004'
 'C-999998' 'C-2284960' 'C-1483744' 'C-864706' 'C-1373463']. At least one non-missing value is needed for imputation with strategy='constant'.
  warnings.warn(


In [72]:
y_pred=modelo.predict(X_test_seleccion)

In [73]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.92      0.95      0.94       613
           1       0.87      0.78      0.82       232

    accuracy                           0.91       845
   macro avg       0.89      0.87      0.88       845
weighted avg       0.90      0.91      0.90       845



In [74]:
from sklearn.metrics import roc_auc_score
y_pred_proba = modelo.predict_proba(X_test_seleccion)[:, 1]

auc_test = roc_auc_score(y_test, y_pred_proba)
print(f"AUC en Test: {auc_test}")

AUC en Test: 0.9555570399954999
